In [46]:
file_path = 'data_parsed_1/NCT03864393.txt'
output_path = 'data_parsed_2/NCT03864393.json'

In [47]:
import re
import json
import json
import os
import matplotlib.pyplot as plt
import networkx as nx
import json
import re
import json
from graphviz import Digraph
import matplotlib.pyplot as plt
from PIL import Image

In [48]:
def read_txt_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    return content

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4)

def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

In [49]:
import re


class Node:
    def __init__(self, operator=None, criteria=None):
        self.operator = operator
        self.criteria = criteria
        self.left = None
        self.right = None

    def to_dict(self):
        if self.operator:
            children_dict = {}
            if self.left:
                children_dict["left"] = self.left.to_dict()
            if self.right:
                children_dict["right"] = self.right.to_dict()
            return {self.operator: children_dict}
        else:
            return {"raw_text": self.criteria if self.criteria is not None else "empty set"}

def parse_criteria(text):
    criteria = []
    lines = text.split('\n')
    for line in lines:
        line = line.strip()
        if line:
            criteria.append(line)
    return criteria

def build_tree(inclusion_criteria, exclusion_criteria):
    root = Node(operator="AND")

    # Build inclusion criteria tree
    inclusion_root = None
    for criterion in inclusion_criteria:
        node = Node(criteria=criterion)
        if inclusion_root is None:
            inclusion_root = node
        else:
            new_root = Node(operator="AND")
            new_root.left = inclusion_root
            new_root.right = node
            inclusion_root = new_root
    root.left = inclusion_root

    # Build exclusion criteria tree
    exclusion_root = None
    for criterion in exclusion_criteria:
        node = Node(criteria=criterion)
        if "[OR]" in criterion:
            substrings = [s.strip() for s in criterion.split("[OR]")]
            or_node = Node(operator="OR")
            or_node.left = Node(criteria=substrings[0])
            or_node.right = Node(criteria=substrings[1])
            node = or_node
        if exclusion_root is None:
            exclusion_root = node
        else:
            new_root = Node(operator="NOT OR")
            new_root.left = exclusion_root
            new_root.right = node
            exclusion_root = new_root
    root.right = exclusion_root

    return root

data = read_json()

inclusion_text, exclusion_text = data.split("Exclusion Criteria:")
inclusion_criteria = parse_criteria(inclusion_text)
exclusion_criteria = parse_criteria(exclusion_text)

# Baum erstellen
tree = build_tree(inclusion_criteria, exclusion_criteria)

# Baum in JSON-Format konvertieren
json_output = json.dumps(tree.to_dict(), indent=2, ensure_ascii=False)

# JSON-Ausgabe in Datei schreiben
with open("output.json", "w", encoding="utf-8") as file:
    file.write(json_output)

In [52]:
from graphviz import Digraph
from PIL import Image
import matplotlib.pyplot as plt
import json

def parse_logic_to_tree(logic, graph, parent=None, node_id=0):
    if 'raw_text' in logic:
        node_label = logic['raw_text']
        graph.node(str(node_id), label=node_label, shape='box')
        if parent is not None:
            graph.edge(parent, str(node_id))
        return node_id

    for key in logic:
        if key in ('AND', 'OR', 'NOT OR'):
            operator = key
            node_label = operator
            current_node_id = node_id
            color = 'lightblue' if operator == 'AND' else 'lightgreen' if operator == 'OR' else 'lightcoral'
            graph.node(str(current_node_id), label=node_label, color=color, fontcolor='black', style='filled', fillcolor=color)
            if parent is not None:
                graph.edge(parent, str(current_node_id))

            operands = logic[key]
            node_id += 1
            if isinstance(operands, dict):
                for child_key in operands:
                    child_id = parse_logic_to_tree(operands[child_key], graph, str(current_node_id), node_id)
                    node_id = child_id + 1
            elif isinstance(operands, list):
                for child in operands:
                    child_id = parse_logic_to_tree(child, graph, str(current_node_id), node_id)
                    node_id = child_id + 1
    return node_id

def visualize_logic_tree(logic):
    graph = Digraph(format='png')
    parse_logic_to_tree(logic, graph)
    return graph

def plot_and_save_graph(logic, graph_title, output_filename):
    graph = visualize_logic_tree(logic)
    graph.render(filename='temp', view=False, cleanup=True)
    image = Image.open('temp.png')
    plt.figure(figsize=(20, 15))
    plt.imshow(image)
    plt.axis('off')
    plt.title(graph_title)
    plt.savefig(output_filename)
    plt.show()

def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data

# Pfad zur JSON-Datei
file_path = 'output.json'

# JSON-Daten lesen
logic_tree = read_json(file_path)

# Baumstruktur visualisieren und speichern
plot_and_save_graph(logic_tree, "Tree Structure Plot", "NCT03860012_tree_plot.png")

In [3]:
import os
import json
from graphviz import Digraph
from PIL import Image
import matplotlib.pyplot as plt
import os
import json
import re
import shutil

class Node:
    def __init__(self, operator=None, criteria=None, left=None, right=None):
        self.operator = operator
        self.criteria = criteria
        self.left = left
        self.right = right

    def to_dict(self):
        if self.criteria:
            return {"raw_text": self.criteria}
        else:
            result = {}
            if self.left:
                result["left"] = self.left.to_dict()
            if self.right:
                result["right"] = self.right.to_dict()
            return {self.operator: result}

def parse_criteria(text, is_inclusion):
    sentences = [sentence.strip() for sentence in text.split('.') if sentence.strip()]
    operator = 'AND' if is_inclusion else 'NOT OR'
    return build_tree(sentences, operator)

def build_tree(sentences, operator):
    if len(sentences) == 1:
        return Node(criteria=sentences[0])
    else:
        mid = len(sentences) // 2
        left_subtree = build_tree(sentences[:mid], operator)
        right_subtree = build_tree(sentences[mid:], operator)
        return Node(operator=operator, left=left_subtree, right=right_subtree)

def parse_logic_to_tree(logic, graph, parent=None, node_id=0):
    if 'raw_text' in logic:
        node_label = logic['raw_text']
        graph.node(str(node_id), label=node_label, shape='box')
        if parent is not None:
            graph.edge(parent, str(node_id))
        return node_id

    for key in logic:
        if key in ('AND', 'OR', 'NOT OR'):
            operator = key
            node_label = operator
            current_node_id = node_id
            color = 'lightblue' if operator == 'AND' else 'lightgreen' if operator == 'OR' else 'lightcoral'
            graph.node(str(current_node_id), label=node_label, color=color, fontcolor='black', style='filled', fillcolor=color)
            if parent is not None:
                graph.edge(parent, str(current_node_id))

            operands = logic[key]
            node_id += 1
            if 'left' in operands:
                left_id = parse_logic_to_tree(operands['left'], graph, str(current_node_id), node_id)
                node_id = left_id + 1
            if 'right' in operands:
                right_id = parse_logic_to_tree(operands['right'], graph, str(current_node_id), node_id)
                node_id = right_id + 1
    return node_id

def visualize_logic_tree(logic):
    graph = Digraph(format='png')
    parse_logic_to_tree(logic, graph)
    return graph

def plot_and_save_graph(logic, graph_title, output_filename):
    graph = visualize_logic_tree(logic)
    graph.render(filename='temp', view=False, cleanup=True)
    image = Image.open('temp.png')
    plt.figure(figsize=(20, 15))
    plt.imshow(image)
    plt.axis('off')
    plt.title(graph_title)
    plt.savefig(output_filename)
    plt.close()

def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return data

def process_files(input_folder, output_folder, plots_folder):
    # Erstelle die Ausgabeordner, falls sie nicht existieren
    os.makedirs(output_folder, exist_ok=True)
    os.makedirs(plots_folder, exist_ok=True)

    # Durchlaufe alle Dateien im Eingabeordner
    for filename in os.listdir(input_folder):
        if filename.endswith(".txt"):
            input_file = os.path.join(input_folder, filename)
            output_file = os.path.join(output_folder, os.path.splitext(filename)[0] + ".json")
            plot_file = os.path.join(plots_folder, os.path.splitext(filename)[0] + "_tree_plot.png")

            # Lese die Eingabedatei
            with open(input_file, 'r', encoding='utf-8') as file:
                data = file.read()

            # Überprüfe, ob Inclusion Criteria oder Exclusion Criteria im ersten Satz vorkommen
            first_sentence = data.split('.')[0].lower()
            if "inclusion criteria" in first_sentence:
                tree = parse_criteria(data, is_inclusion=True)
            elif "exclusion criteria" in first_sentence:
                tree = parse_criteria(data, is_inclusion=False)
            else:
                print(f"Überspringe Datei {filename}, da weder Inclusion Criteria noch Exclusion Criteria im ersten Satz gefunden wurden.")
                shutil.copy(input_file, sonderfälle_folder)
                continue

            # Baum in JSON-Format konvertieren
            json_output = json.dumps(tree.to_dict(), indent=2, ensure_ascii=False)

            # JSON-Ausgabe in Datei schreiben
            with open(output_file, 'w', encoding='utf-8') as file:
                file.write(json_output)

            # JSON-Daten lesen
            logic_tree = read_json(output_file)

            # Baumstruktur visualisieren und speichern
            #plot_and_save_graph(logic_tree, "Tree Structure Plot", plot_file)

sonderfälle_folder = "sonderfälle"
input_folder = "data_parsed_1"
output_folder = "data_parsed_2"
plots_folder = "plots"

# Verarbeite alle Dateien
process_files(input_folder, output_folder, plots_folder)